# RAG asistido por recuperación — SUNAT

### Respuestas ancladas a documentación oficial

**Objetivo de esta demo** · Mostrar, en pocos minutos, cómo el sistema **responde a preguntas en lenguaje natural** usando **evidencia real** (gob.pe, SUNAT, cartillas en PDF).

> *“El modelo no inventa de memoria: prioriza lo que recuperamos del corpus tributario.”*

---


### Antes de ejecutar (Colab)

Sube a **`data/processed/`** los reportes `*.json` del pipeline y la carpeta **`faiss_saved/`** (`faiss_hnsw.index` + `faiss_hnsw_metadata.json`) si harás la demo en vivo.

Si aparece **`ModuleNotFoundError: sunat_faiss_runtime`**, el clon en `/content/...` suele estar desactualizado: **Entorno → Reiniciar y ejecutar todo**, o `!git -C /content/rag-mds2025-ia-proyectofinal pull`.


## 1 · Entorno

Preparamos el repositorio y las dependencias *una sola vez*; el resto del notebook solo **lee** artefactos ya generados.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def find_project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(5):
        if (p / "requirements.txt").is_file() and (p / "src").is_dir():
            return p
        p = p.parent
    raise FileNotFoundError("Raíz del repo no encontrada (busca requirements.txt + src/).")


def ensure_project_root_for_imports() -> Path:
    """Garantiza sys.path y RAG_PROJECT_ROOT; exige sunat_faiss_runtime.py (Colab/local)."""
    candidates: list[Path] = []
    env_root = os.environ.get("RAG_PROJECT_ROOT")
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend(
        [
            Path("/content/rag-mds2025-ia-proyectofinal"),
            Path.cwd().resolve(),
            Path.cwd().resolve().parent,
            Path.cwd().resolve().parent.parent,
        ]
    )
    seen: set[Path] = set()
    for raw in candidates:
        root = raw.resolve()
        if root in seen:
            continue
        seen.add(root)
        marker = root / "src" / "interfaces" / "sunat_faiss_runtime.py"
        if marker.is_file():
            rs = str(root)
            if rs not in sys.path:
                sys.path.insert(0, rs)
            os.environ["RAG_PROJECT_ROOT"] = rs
            return root
    raise ImportError(
        "No se encuentra src/interfaces/sunat_faiss_runtime.py. "
        "Ejecuta: !git -C /content/rag-mds2025-ia-proyectofinal pull origin main "
        "y reinicia el runtime."
    )


IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/junvegu/rag-mds2025-ia-proyectofinal.git"
CLONE_DIR = Path("/content/rag-mds2025-ia-proyectofinal")

if IN_COLAB and not CLONE_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(CLONE_DIR)],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
if IN_COLAB:
    os.chdir(CLONE_DIR)

# Actualizar repo si ya existía (sesiones viejas sin sunat_faiss_runtime.py)
if IN_COLAB and CLONE_DIR.is_dir() and (CLONE_DIR / ".git").exists():
    subprocess.run(
        ["git", "-C", str(CLONE_DIR), "pull", "--ff-only"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
os.environ["RAG_PROJECT_ROOT"] = str(PROJECT_ROOT)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

from src.config.runtime_env import apply_darwin_openmp_mitigations

apply_darwin_openmp_mitigations()
PROJECT_ROOT = ensure_project_root_for_imports()
print("✓  Entorno listo  ·  ", PROJECT_ROOT)


### Utilidades de salida

Ejecuta la **siguiente celda** antes de la sección 2 (define tablas, paneles y tarjetas).


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

PROJECT_ROOT = ensure_project_root_for_imports()

# --- Rutas fijas ---
PROCESSED: Path = PROJECT_ROOT / "data" / "processed"
EVAL_PATH: Path = PROCESSED / "evaluation_report.json"
FAISS_DIR: Path = PROCESSED / "faiss_saved"

REQUIRED_REPORTS: list[str] = [
    "ingestion_report.json",
    "chunking_report.json",
    "embeddings_report.json",
    "faiss_report.json",
    "hybrid_retrieval_report.json",
    "reranker_report.json",
    "generation_report.json",
    "evaluation_report.json",
]


def _rule(char: str = "─", width: int = 58) -> str:
    return char * width


def banner(title: str, subtitle: str | None = None) -> None:
    print()
    print(_rule("═"))
    print(f"  {title}")
    if subtitle:
        print(f"  {subtitle}")
    print(_rule("═"))


def truncate_text(text: str, max_chars: int = 900, placeholder: str = " …") -> str:
    t = (text or "").strip()
    if len(t) <= max_chars:
        return t
    n = max(0, max_chars - len(placeholder))
    return t[:n] + placeholder


def print_artifacts_status(processed: Path, names: list[str]) -> None:
    banner("Artefactos detectados", "¿Tenemos trazabilidad del pipeline guardada?")
    print(f"\n  {'Archivo':<40}  Estado")
    print(f"  {_rule('·', 52)}")
    for name in names:
        ok = (processed / name).is_file()
        icon = "✅" if ok else "❌"
        label = "presente" if ok else "ausente"
        print(f"  {icon}  {name:<36}  {label}")
    n_ok = sum(1 for n in names if (processed / n).is_file())
    print(f"\n  →  {n_ok}/{len(names)} archivos listos\n")


def print_metrics_dashboard(eval_path: Path) -> bool:
    if not eval_path.is_file():
        banner("Métricas", "No hay evaluation_report.json")
        print("  Sube el archivo a data/processed/ para ver el panel.\n")
        return False
    rep = json.loads(eval_path.read_text(encoding="utf-8"))
    gm = rep.get("global_metrics") or {}
    con = rep.get("conclusion") or {}
    banner("Resumen ejecutivo · evaluación automática", "Métricas sobre el set de preguntas de referencia")
    verdict = con.get("verdict_es", "—")
    summary = con.get("summary_es", "")
    print(f"\n  {'Conclusión automática':<22}  {verdict}")
    if summary:
        print(f"\n  {truncate_text(summary, 320)}\n")
    print(f"  {_rule('─', 52)}")
    print(f"  {'avg_rouge_1':<22}  {gm.get('avg_rouge_1', '—')}")
    print(f"  {'avg_rouge_l':<22}  {gm.get('avg_rouge_l', '—')}")
    print(f"  {'avg_grounding_score':<22}  {gm.get('avg_grounding_score', '—')}")
    print(f"  {'total_hallucination_flags':<22}  {gm.get('total_hallucination_flags', '—')}")
    print(f"  {_rule('─', 52)}\n")
    return True


def print_example_card(idx: int, row: dict, max_answer: int = 900) -> None:
    qid = row.get("question_id", "")
    print(f"\n  ▸  EJEMPLO {idx}  ·  {qid}")
    print(f"  {_rule('─', 54)}")
    print("  Pregunta")
    q = truncate_text(row.get("question") or "", 520)
    for ln in q.split("\n")[:5]:
        print(f"     {ln}")
    print(f"\n  {_rule('·', 54)}")
    print("  Respuesta generada")
    a = truncate_text(row.get("generated_answer") or "", max_answer)
    for ln in a.split("\n")[:10]:
        if ln.strip():
            print(f"     {ln}")
    print(f"\n  Métricas    grounding: {row.get('grounding_score')}    ROUGE-L: {row.get('rouge_l')}")
    print("  Fuentes")
    for s in (row.get("sources_used") or [])[:5]:
        print(f"     • {truncate_text(s, 68)}")
    print(f"  {_rule('─', 54)}")


def print_live_demo_block(question: str, ans) -> None:
    banner("Resultado en vivo", "Índice guardado + BM25 + híbrido + rerank + Qwen + citas")
    print(f"\n  Pregunta\n  {_rule('─', 44)}\n  {truncate_text(question, 600)}\n")
    print(f"  Respuesta\n  {_rule('─', 40)}\n  {truncate_text(ans.text, 2200)}\n")
    print(f"  {_rule('─', 40)}")
    print(f"  grounding_score      {ans.grounding_score}")
    print(f"  hallucination_flag   {ans.hallucination_flag}\n")
    banner("Evidencia citada (3 fragmentos)", "Oración · fuente · página")
    for j, c in enumerate((ans.citations or [])[:3], 1):
        print(f"\n  [{j}]  {truncate_text(c.sentence, 200)}")
        print(f"       fuente · {truncate_text(c.source or '', 70)}")
        print(f"       página · {c.page}\n")


def print_executive_snapshot(resumen: dict) -> None:
    banner("Cierre · snapshot del sistema", "Listo para captura / slides")
    wk = max(len(str(k)) for k in resumen)
    for k, v in resumen.items():
        print(f"  {str(k):<{wk}}    {v}")
    print()


## 2 · Estado del sistema

**Qué validamos** · Que el pipeline previo dejó **huella en disco** (reportes por fase).

**Por qué importa** · Sin artefactos no hay **historia auditable** ni demo reproducible en Colab.


In [ ]:
print_artifacts_status(PROCESSED, REQUIRED_REPORTS)


## 3 · Métricas finales

**Qué mostramos** · Cómo se comporta el sistema frente a **respuestas de referencia** (ROUGE) y **anclaje al contexto** (grounding).

**Por qué importa** · Cierra el ciclo **cuantitativo** además de la demo cualitativa.


In [ ]:
print_metrics_dashboard(EVAL_PATH)


## 4 · Dos respuestas del evaluation set

**Qué mostramos** · Preguntas reales, texto generado y **de dónde salió** la evidencia.

**Por qué importa** · La **trazabilidad** es el argumento central frente a un LLM “suelto”.


In [ ]:
if not EVAL_PATH.is_file():
    banner("Ejemplos", "Falta evaluation_report.json")
else:
    rep = json.loads(EVAL_PATH.read_text(encoding="utf-8"))
    items = rep.get("by_question") or []
    if len(items) < 2:
        print("  No hay al menos 2 ítems en by_question.\n")
    else:
        for i, row in enumerate(items[:2], 1):
            print_example_card(i, row)
        print()


## 5 · Demo en vivo

**Qué hace esta celda** · Carga el **índice FAISS** ya guardado, reconstruye solo **BM25 en RAM** desde metadatos, y ejecuta **híbrido → rerank → Qwen → citas**.

**Qué no hace** · No vuelve a ingerir la web ni a embedder los 500+ chunks.

**Por qué importa** · Demuestra **latencia y calidad** en condiciones realistas de reutilización.


In [ ]:
# Por si el kernel no siguió el orden: re-localiza el repo y sys.path
PROJECT_ROOT = ensure_project_root_for_imports()
FAISS_DIR = PROJECT_ROOT / "data" / "processed" / "faiss_saved"

from src.interfaces.sunat_faiss_runtime import answer_from_saved_faiss

DEMO_QUESTION = "¿Cómo se calcula el impuesto a la renta de quinta categoría?"
need = [FAISS_DIR / "faiss_hnsw.index", FAISS_DIR / "faiss_hnsw_metadata.json"]

if not all(p.is_file() for p in need):
    banner("Demo en vivo", "Índice no disponible")
    print("  Coloca en  data/processed/faiss_saved/  los archivos:")
    print("    · faiss_hnsw.index")
    print("    · faiss_hnsw_metadata.json")
    print()
    print("  En máquina local (una vez):")
    print("    python scripts/test_faiss_hnsw.py --index-dir data/processed/faiss_saved")
    print()
else:
    ans = answer_from_saved_faiss(
        DEMO_QUESTION,
        faiss_dir=FAISS_DIR,
        top_k=5,
        hybrid_pool=10,
    )
    print_live_demo_block(DEMO_QUESTION, ans)


## 6 · Cierre ejecutivo

**Qué es** · Una **foto única** del sistema (corpus, modelos, métricas agregadas).

**Para qué** · Cierre oral, **screenshot** o transcripción a diapositivas.


In [ ]:
def _load_json(path: Path):
    if not path.is_file():
        return None
    return json.loads(path.read_text(encoding="utf-8"))


ing = _load_json(PROCESSED / "ingestion_report.json")
chk = _load_json(PROCESSED / "chunking_report.json")
fax = _load_json(PROCESSED / "faiss_report.json")
evl = _load_json(EVAL_PATH)

ing_r = (ing or {}).get("report") or {}
chk_r = (chk or {}).get("report") or {}
fax_r = (fax or {}).get("report") or {}
gm = (evl or {}).get("global_metrics") or {}
ex = (evl or {}).get("executive_summary") or {}
models = ex.get("models_and_index") or {}

resumen = {
    "fuentes": ing_r.get("total_sources"),
    "documentos": ing_r.get("total_documents") or chk_r.get("input_documents"),
    "chunks": chk_r.get("generated_chunks") or fax_r.get("total_chunks"),
    "modelo_embeddings": (evl or {}).get("embedding_model") or models.get("embedding_model"),
    "indice": "FAISS HNSW",
    "retrieval": "Hybrid + Reranker",
    "LLM": (evl or {}).get("generation_model") or models.get("generation_model") or "Qwen",
    "grounding_promedio": gm.get("avg_grounding_score"),
    "rouge_1": gm.get("avg_rouge_1"),
    "rouge_l": gm.get("avg_rouge_l"),
}

print_executive_snapshot(resumen)
